In [1]:
real = spark.read.format("delta").load(
"abfss://curated@energybigdatastorage.dfs.core.windows.net/ml_ready_dataset/"
)

pred = spark.read.format("delta").load(
"abfss://curated@energybigdatastorage.dfs.core.windows.net/prophet_predictions/"
)

In [2]:
from pyspark.sql.functions import col
pred = pred.select(
col("ds"),
col("yhat").alias("prediction")
)

In [3]:
final_df = real.join(
pred,
real["tstp"] == pred["ds"],
"left"
)

In [4]:
from pyspark.sql.functions import abs

final_df = final_df.withColumn(
"error",
abs(col("energy_kwh") - col("prediction"))
)

In [5]:
from pyspark.sql.functions import hour, when
final_df = final_df.withColumn(
"hour",
hour("tstp")
).withColumn(
"is_peak",
when((col("hour")>=7)&(col("hour")<=10),1)
.when((col("hour")>=17)&(col("hour")<=21),1)
.otherwise(0)
)

In [6]:
final_df.write.format("delta") \
.mode("overwrite") \
.save(
"abfss://curated@energybigdatastorage.dfs.core.windows.net/powerbi_final/"
)

In [7]:
print("PIPELINE OK")

In [8]:
final_df.write.mode("overwrite").option("header", True).csv(
"abfss://curated@energybigdatastorage.dfs.core.windows.net/powerbi_final/"
)

In [9]:
mssparkutils.fs.ls(
"abfss://curated@energybigdatastorage.dfs.core.windows.net/powerbi_final/"
)

In [10]:
df = spark.read.option("header", True).csv(
"abfss://curated@energybigdatastorage.dfs.core.windows.net/powerbi_final/"
)

df.show(5)